# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8180 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

16:40:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [2]:
GPS_DIR = f"{DATA}/vehicules"

schema_produits = T.StructType([
    T.StructField("produit_id",     T.StringType()),
    T.StructField("nom",            T.StringType()),
    T.StructField("categorie",      T.StringType()),
    T.StructField("sous_categorie", T.StringType()),
    T.StructField("prix",           T.DoubleType()),
    T.StructField("poids_kg",       T.DoubleType()),
])

schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType()),
    T.StructField("client_id",   T.StringType()),
    T.StructField("horodatage",  T.TimestampType()),
    T.StructField("evenement",   T.StringType()),
    T.StructField("livreur_id",  T.StringType()),
    T.StructField("detail",      T.StringType()),
])

schema_communes = T.StructType([
    T.StructField("commune",     T.StringType()),
    T.StructField("code_postal", T.StringType()),
    T.StructField("lon",         T.DoubleType()),
    T.StructField("lat",         T.DoubleType()),
])

schema_gps = T.StructType([
    T.StructField("timestamp",     T.TimestampType()),
    T.StructField("livreur_id",    T.StringType()),
    T.StructField("lon",           T.DoubleType()),
    T.StructField("lat",           T.DoubleType()),
    T.StructField("vitesse_kmh",   T.DoubleType()),
    T.StructField("dans_vehicule", T.BooleanType()),
    T.StructField("charge_kg",     T.DoubleType()),
])


def lire_csv(chemin, schema):
    return (spark.read
            .option("header", True)
            .option("escape", '"')
            .schema(schema)
            .csv(chemin))


produits = lire_csv(f"{DATA}/produits.csv", schema_produits).cache()
journal  = lire_csv(f"{DATA}/commandes.csv", schema_commandes).cache()
communes = lire_csv(f"{DATA}/communes.csv", schema_communes).cache()
gps      = lire_csv(f"{GPS_DIR}/*/*.csv", schema_gps)

for nom, df in [("produits", produits), ("commandes (journal)", journal),
                ("communes", communes), ("gps", gps)]:
    print(f"{nom:20s} {df.count():>12,d} lignes")

produits                      233 lignes


commandes (journal)       846,535 lignes
communes                      142 lignes


gps                     5,174,905 lignes


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [3]:
gps = (gps
       .withColumn("vehicule_id", F.regexp_extract(F.input_file_name(), r"/([^/]+)/[^/]+\.csv$", 1))
       .withColumn("jour", F.to_date("timestamp")))
_cache_gps = {"df": None}
def cache_gps(df):
    df = df.cache()
    df.count()
    if _cache_gps["df"] is not None:
        _cache_gps["df"].unpersist() 
    _cache_gps["df"] = df
    return df

gps = cache_gps(gps)
gps.show(5, truncate=False)
gps.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c)
            for c in ["timestamp", "vehicule_id", "dans_vehicule", "lon", "lat"]]).show()

+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|timestamp          |livreur_id|lon     |lat      |vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|jour      |
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|2025-10-01 07:40:00|L001      |2.277083|48.936735|0.0        |false        |2581.6   |V18        |2025-10-01|
|2025-10-01 07:41:00|L001      |2.277083|48.936735|0.0        |false        |2581.6   |V18        |2025-10-01|
|2025-10-01 07:42:00|L001      |2.277084|48.936734|0.0        |false        |2581.6   |V18        |2025-10-01|
|2025-10-01 07:43:00|L001      |2.277083|48.936734|0.0        |false        |2581.6   |V18        |2025-10-01|
|2025-10-01 07:44:00|L001      |2.277084|48.936733|0.0        |false        |2581.6   |V18        |2025-10-01|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
o

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [4]:
journal.agg(
    F.countDistinct("commande_id").alias("commandes"),
    F.countDistinct("client_id").alias("clients"),
    F.countDistinct("livreur_id").alias("livreurs"),
).show()

gps.agg(
    F.countDistinct("vehicule_id").alias("vehicules"),
    F.countDistinct("livreur_id").alias("livreurs_dans_le_gps"),
).show()

+---------+-------+--------+
|commandes|clients|livreurs|
+---------+-------+--------+
|   271617|   8000|      90|
+---------+-------+--------+



+---------+--------------------+
|vehicules|livreurs_dans_le_gps|
+---------+--------------------+
|       72|                  90|
+---------+--------------------+



2.2) Comptez les événements du journal par type.

In [5]:
journal.groupBy("evenement").count().orderBy(F.desc("count")).show()

+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|   livree|268682|
|    echec| 11220|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [6]:
nb_chargee = (journal
              .groupBy("commande_id")
              .agg(F.count(F.when(F.col("evenement") == "chargee", 1)).alias("nb_chargee")))

nb_chargee.groupBy("nb_chargee").count().orderBy("nb_chargee").show()


+----------+------+
|nb_chargee| count|
+----------+------+
|         0|  1637|
|         1|254229|
|         2| 14673|
|         3|  1026|
|         4|    51|
|         5|     1|
+----------+------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [7]:
ordre = (F.when(F.col("evenement") == "commande", 0)
          .when(F.col("evenement") == "chargee", 1)
          .when(F.col("evenement") == "retour", 3)
          .otherwise(2))
w_cmd = Window.partitionBy("commande_id").orderBy("horodatage", ordre)

precedents = (journal
              .withColumn("evenement_precedent", F.lag("evenement").over(w_cmd))
              .filter(F.col("evenement") == "chargee"))

precedents.groupBy("evenement_precedent").count().orderBy(F.desc("count")).show()

+-------------------+------+
|evenement_precedent| count|
+-------------------+------+
|           commande|269980|
|              echec|  9950|
|           reportee|  6932|
+-------------------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [8]:
etat = (journal
        .filter(F.col("horodatage") < "2026-01-01")
        .groupBy("commande_id")
        .agg(F.max((F.col("evenement") == "livree").cast("int")).alias("livree"),
             F.max((F.col("evenement") == "retour").cast("int")).alias("retour")))

non_livrees = (etat
               .filter(F.col("livree") == 0)
               .withColumn("statut", F.when(F.col("retour") == 1, "abandonnee (retour)")
                                      .otherwise("encore en cours")))

print("commandes non livrees au 31/12/2025 :", non_livrees.count())
non_livrees.groupBy("statut").count().show(truncate=False)

commandes non livrees au 31/12/2025 : 2935
+-------------------+-----+
|statut             |count|
+-------------------+-----+
|abandonnee (retour)|1194 |
|encore en cours    |1741 |
+-------------------+-----+



## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [9]:
motif_item = r"^(P\d+)x(\d+)$"

lignes = (journal
          .filter(F.col("evenement") == "commande")
          .select("commande_id", "client_id", "horodatage",
                  F.explode(F.split("detail", r"\|")).alias("item"))
          .withColumn("produit_id", F.regexp_extract("item", motif_item, 1))
          .withColumn("quantite",   F.regexp_extract("item", motif_item, 2).cast("int"))
          .drop("item"))

lignes.show(5)
print("lignes de commande :", lignes.count())

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0037|       5|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0063|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0019|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0067|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0069|       2|
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows

lignes de commande : 4779049


3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [10]:
lignes = (lignes
          .join(F.broadcast(produits), "produit_id", "left")
          .withColumn("montant",     F.round(F.col("prix") * F.col("quantite"), 2))
          .withColumn("poids_ligne", F.col("poids_kg") * F.col("quantite"))
          .cache())
print("lignes sans produit dans le catalogue :", lignes.filter(F.col("prix").isNull()).count())
lignes.show(5, truncate=False)

commandes = (lignes
             .groupBy("commande_id", "client_id", "horodatage")
             .agg(F.round(F.sum("montant"), 2).alias("montant"),
                  F.round(F.sum("poids_ligne"), 3).alias("poids_kg"),
                  F.count("*").alias("nb_lignes"))
             .cache())
commandes.show(5)
print("commandes :", commandes.count())

lignes sans produit dans le catalogue : 0
+----------+-----------+---------+-------------------+--------+----------------------+---------+-------------------+----+--------+-------+-----------+
|produit_id|commande_id|client_id|horodatage         |quantite|nom                   |categorie|sous_categorie     |prix|poids_kg|montant|poids_ligne|
+----------+-----------+---------+-------------------+--------+----------------------+---------+-------------------+----+--------+-------+-----------+
|P0037     |C0000001   |CL05658  |2025-01-01 00:03:44|5       |Cookies chocolat 300 g|Épicerie |Biscuits et snacks |2.6 |0.3     |13.0   |1.5        |
|P0063     |C0000001   |CL05658  |2025-01-01 00:03:44|1       |Lardons fumés 200 g   |Frais    |Viandes et poissons|2.4 |0.2     |2.4    |0.2        |
|P0019     |C0000001   |CL05658  |2025-01-01 00:03:44|2       |Capsules café x30     |Épicerie |Petit-déjeuner     |8.5 |0.2     |17.0   |0.4        |
|P0067     |C0000001   |CL05658  |2025-01-01 00:03:4

+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|         horodatage|montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|   C0000168|  CL06333|2025-01-01 13:16:07| 209.35|   60.98|       32|
|   C0000469|  CL03079|2025-01-01 22:39:48|   80.9|   14.03|       13|
|   C0000752|  CL00230|2025-01-02 16:34:53| 218.05|  46.515|       31|
|   C0000810|  CL07244|2025-01-02 18:29:22| 239.45|    78.5|       27|
|   C0000855|  CL04907|2025-01-02 19:48:36|   70.9|   15.87|       15|
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows



commandes : 271617


## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [11]:
bornes = (gps.groupBy("vehicule_id", "jour")
          .agg(F.min_by("lon", "timestamp").alias("lon_premier"),
               F.min_by("lat", "timestamp").alias("lat_premier"),
               F.max_by("lon", "timestamp").alias("lon_dernier"),
               F.max_by("lat", "timestamp").alias("lat_dernier"))
          .cache())

points = (bornes.select("vehicule_id", "jour", F.lit("premier").alias("point"),
                        F.col("lon_premier").alias("lon"), F.col("lat_premier").alias("lat"))
          .unionByName(
              bornes.select("vehicule_id", "jour", F.lit("dernier").alias("point"),
                            F.col("lon_dernier").alias("lon"), F.col("lat_dernier").alias("lat")))
          .withColumn("lon_r", F.round("lon", 2))
          .withColumn("lat_r", F.round("lat", 2)))

positions = (points.groupBy("lon_r", "lat_r")
             .pivot("point", ["premier", "dernier"]).count()
             .na.fill(0)
             .withColumn("total", F.col("premier") + F.col("dernier"))
             .orderBy(F.desc("total")))
positions.show(20)

nb_journees = bornes.count()
SEUIL = 0.005 * nb_journees
nb_entrepots = positions.filter(F.col("premier") >= SEUIL).count()
print(f"{nb_journees} journées-véhicule -> {nb_entrepots} entrepôts")

+-----+-----+-------+-------+-----+
|lon_r|lat_r|premier|dernier|total|
+-----+-----+-------+-------+-----+
| 2.45|48.91|   4217|   4217| 8434|
| 2.28|48.94|   3208|   3208| 6416|
| 2.35|48.75|   2967|   2967| 5934|
+-----+-----+-------+-------+-----+



10392 journées-véhicule -> 3 entrepôts


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [12]:
entrepots = (points.filter(F.col("point") == "premier")
             .groupBy("lon_r", "lat_r")
             .agg(F.avg("lon").alias("lon"), F.avg("lat").alias("lat"), F.count("*").alias("nb_jours"))
             .filter(F.col("nb_jours") >= SEUIL)
             .withColumn("entrepot",
                         F.concat(F.lit("E"), F.row_number().over(Window.orderBy("lon")).cast("string")))
             .select("entrepot", "lon", "lat")
             .cache())
entrepots.show()

ent_b = F.broadcast(entrepots.select("entrepot", F.col("lon").alias("e_lon"), F.col("lat").alias("e_lat")))
w_proche = Window.partitionBy("vehicule_id", "jour").orderBy("d_km")
w_freq   = Window.partitionBy("vehicule_id").orderBy(F.desc("nb_jours"))

vehicule_entrepot = (points.filter(F.col("point") == "premier")
                     .crossJoin(ent_b)
                     .withColumn("d_km", haversine("lon", "lat", "e_lon", "e_lat"))
                     .withColumn("rn", F.row_number().over(w_proche)).filter(F.col("rn") == 1)
                     .groupBy("vehicule_id", "entrepot").agg(F.count("*").alias("nb_jours"))
                     .withColumn("rn", F.row_number().over(w_freq)).filter(F.col("rn") == 1)
                     .select("vehicule_id", "entrepot")
                     .cache())
vehicule_entrepot.groupBy("entrepot").agg(F.count("*").alias("nb_vehicules")).orderBy("entrepot").show()

16:42:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:42:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:42:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:42:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:42:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:42:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:42:42 WARN WindowExec: No Partition Defined for Window operat

+--------+------------------+------------------+
|entrepot|               lon|               lat|
+--------+------------------+------------------+
|      E1| 2.277068913341647|48.936712904613465|
|      E2| 2.351513594202898| 48.74704698584426|
|      E3|2.4489591059995255|48.906701645482585|
+--------+------------------+------------------+



+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|      E1|          24|
|      E2|          24|
|      E3|          24|
+--------+------------+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [13]:
w_lj = Window.partitionBy("livreur_id", "jour").orderBy("timestamp")

gps = gps.withColumn(
    "changement",
    F.when(F.lag("dans_vehicule").over(w_lj) & ~F.col("dans_vehicule"), 1).otherwise(0))

(gps.filter((F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06") & (F.col("changement") == 1))
    .select("timestamp", "vehicule_id", "lon", "lat", "dans_vehicule", "charge_kg")
    .orderBy("timestamp")
    .show(100, truncate=False))

+-------------------+-----------+--------+---------+-------------+---------+
|timestamp          |vehicule_id|lon     |lat      |dans_vehicule|charge_kg|
+-------------------+-----------+--------+---------+-------------+---------+
|2025-01-06 09:54:00|V10        |2.311953|48.871389|false        |1143.9   |
|2025-01-06 10:00:00|V10        |2.313447|48.870848|false        |1089.1   |
|2025-01-06 10:09:00|V10        |2.316012|48.872919|false        |1085.5   |
|2025-01-06 10:23:00|V10        |2.325412|48.874465|false        |959.0    |
|2025-01-06 10:36:00|V10        |2.32844 |48.866695|false        |916.7    |
|2025-01-06 10:54:00|V10        |2.337183|48.866435|false        |915.7    |
|2025-01-06 11:12:00|V10        |2.317523|48.857452|false        |874.0    |
|2025-01-06 11:38:00|V10        |2.317717|48.850169|false        |840.6    |
|2025-01-06 12:02:00|V10        |2.312056|48.845401|false        |530.9    |
|2025-01-06 12:15:00|V10        |2.305215|48.854413|false        |496.5    |

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [14]:
cumul = w_lj.rowsBetween(Window.unboundedPreceding, Window.currentRow)
gps = cache_gps(gps.withColumn("numero_livraison", F.sum("changement").over(cumul)))

(gps.filter((F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06"))
    .groupBy("numero_livraison")
    .agg(F.min("timestamp").alias("premier_point"),
         F.max("timestamp").alias("dernier_point"),
         F.count(F.when(~F.col("dans_vehicule"), 1)).alias("minutes_a_pied"))
    .orderBy("numero_livraison")
    .show(100, truncate=False))

+----------------+-------------------+-------------------+--------------+
|numero_livraison|premier_point      |dernier_point      |minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|0               |2025-01-06 07:43:00|2025-01-06 09:53:00|44            |
|1               |2025-01-06 09:54:00|2025-01-06 09:59:00|4             |
|2               |2025-01-06 10:00:00|2025-01-06 10:08:00|7             |
|3               |2025-01-06 10:09:00|2025-01-06 10:22:00|10            |
|4               |2025-01-06 10:23:00|2025-01-06 10:35:00|4             |
|5               |2025-01-06 10:36:00|2025-01-06 10:53:00|8             |
|6               |2025-01-06 10:54:00|2025-01-06 11:11:00|4             |
|7               |2025-01-06 11:12:00|2025-01-06 11:37:00|18            |
|8               |2025-01-06 11:38:00|2025-01-06 12:01:00|17            |
|9               |2025-01-06 12:02:00|2025-01-06 12:14:00|4             |
|10              |2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [15]:
CLES = ["livreur_id", "jour", "numero_livraison"]

livraisons = (gps.filter(~F.col("dans_vehicule"))
              .groupBy(*CLES)
              .agg(F.max("vehicule_id").alias("vehicule_id"),
                   F.min("timestamp").alias("debut"),
                   F.max("timestamp").alias("fin"),
                   F.count("*").alias("duree_min"),
                   F.avg("lon").alias("lon"),
                   F.avg("lat").alias("lat"))
              .cache())

livraisons.show(5, truncate=False)
print("lignes dans livraisons :", livraisons.count())

+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+
|livreur_id|jour      |numero_livraison|vehicule_id|debut              |fin                |duree_min|lon               |lat               |
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+
|L002      |2025-05-09|0               |V03        |2025-05-09 07:41:00|2025-05-09 08:23:00|43       |2.2770558372093013|48.93673423255813 |
|L002      |2025-05-09|1               |V03        |2025-05-09 10:37:00|2025-05-09 10:41:00|5        |2.3160694000000004|48.8426472        |
|L002      |2025-05-09|2               |V03        |2025-05-09 10:49:00|2025-05-09 10:57:00|9        |2.312650777777778 |48.842491333333335|
|L002      |2025-05-09|3               |V03        |2025-05-09 11:01:00|2025-05-09 11:09:00|9        |2.3084496666666663|48.84163044444445 |
|L002      |2

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [16]:
ev = (journal
      .filter(F.col("evenement").isin("livree", "echec"))
      .select("livreur_id",
              F.to_date("horodatage").alias("jour"),
              F.col("horodatage").alias("signature"),
              "evenement", "commande_id", "client_id"))

l, e = livraisons.alias("l"), ev.alias("e")
marge = F.expr("INTERVAL 1 MINUTE")
condition = ((F.col("l.livreur_id") == F.col("e.livreur_id")) &
             (F.col("l.jour") == F.col("e.jour")) &
             (F.col("e.signature") >= F.col("l.debut") - marge) &
             (F.col("e.signature") <= F.col("l.fin") + marge))

livraisons_ev = (l.join(e, condition, "left")
                 .select("l.*", "e.signature", "e.evenement", "e.commande_id", "e.client_id")
                 .cache())

livraisons_ev.orderBy(*CLES, "signature").show(10, truncate=False)
print("lignes dans livraisons_ev :", livraisons_ev.count())

+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|livreur_id|jour      |numero_livraison|vehicule_id|debut              |fin                |duree_min|lon               |lat               |signature          |evenement|commande_id|client_id|
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|L001      |2025-01-06|0               |V10        |2025-01-06 07:43:00|2025-01-06 08:26:00|44       |2.277131022727271 |48.93670315909091 |NULL               |NULL     |NULL       |NULL     |
|L001      |2025-01-06|1               |V10        |2025-01-06 09:54:00|2025-01-06 09:57:00|4        |2.311953          |48.87138975       |2025-01-06 09:54:50|livree   |C0001606   |CL07994  |
|L001      |2025-01-06|2           

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [17]:
rattaches = (livraisons_ev
             .filter(F.col("commande_id").isNotNull())
             .select("commande_id", "signature", "evenement").distinct())

orphelins = ev.join(rattaches, ["commande_id", "signature", "evenement"], "left_anti")
print("événements livree/echec rattachés à aucune livraison :", orphelins.count())
orphelins.groupBy("evenement").count().show()

print("événements rattachés 2 fois :",
      livraisons_ev.filter(F.col("commande_id").isNotNull())
                   .groupBy("commande_id", "signature").count().filter(F.col("count") > 1).count())

événements livree/echec rattachés à aucune livraison : 0


+---------+-----+
|evenement|count|
+---------+-----+
+---------+-----+



événements rattachés 2 fois : 0


5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

In [18]:
ent_pos = F.broadcast(entrepots.select(F.col("lon").alias("e_lon"), F.col("lat").alias("e_lat")))
dist_entrepot = (livraisons.crossJoin(ent_pos)
                 .withColumn("d_km", haversine("lon", "lat", "e_lon", "e_lat"))
                 .groupBy(*CLES).agg(F.min("d_km").alias("dist_entrepot_km")))

nb_ev = livraisons_ev.groupBy(*CLES).agg(F.count("evenement").alias("nb_ev"))

types = (nb_ev.join(dist_entrepot, CLES)
         .withColumn("type", F.when(F.col("nb_ev") > 0, "livraison")
                              .when(F.col("dist_entrepot_km") < 0.3, "entrepot")
                              .otherwise("pause"))
         .select(*CLES, "type")
         .cache())

livraisons_ev = livraisons_ev.join(types, CLES).cache()

minutes_debut = F.hour("debut") * 60 + F.minute("debut")
(livraisons.join(types, CLES)
    .groupBy("type")
    .agg(F.count("*").alias("nb"),
         F.round(F.avg("duree_min"), 1).alias("duree_moy_min"),
         F.round(F.avg(minutes_debut)).cast("int").alias("m"))
    .withColumn("heure_moy_debut",
                F.format_string("%02d:%02d", (F.col("m") / 60).cast("int"), F.col("m") % 60))
    .drop("m")
    .orderBy(F.desc("nb"))
    .show())

+---------+------+-------------+---------------+
|     type|    nb|duree_moy_min|heure_moy_debut|
+---------+------+-------------+---------------+
|livraison|246258|          8.3|          12:11|
| entrepot| 22368|         27.2|          11:41|
|    pause|  4967|         37.5|          12:21|
+---------+------+-------------+---------------+



5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

In [19]:
par_livraison = (livraisons_ev
                 .filter(F.col("type") == "livraison")
                 .groupBy(*CLES)
                 .agg(F.count("evenement").alias("nb_ev"),
                      F.countDistinct("client_id").alias("nb_clients")))

par_livraison.groupBy("nb_ev").count().orderBy("nb_ev").show()
print("livraisons avec au moins 2 clients différents :",
      par_livraison.filter(F.col("nb_clients") >= 2).count())

+-----+------+
|nb_ev| count|
+-----+------+
|    1|217538|
|    2| 24873|
|    3|  3053|
|    4|   592|
|    5|   155|
|    6|    29|
|    7|     8|
|    8|     5|
|    9|     4|
|   10|     1|
+-----+------+



livraisons avec au moins 2 clients différents : 12536


5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

In [20]:
w_ev = Window.partitionBy(*CLES).orderBy("signature")
depuis = F.coalesce(F.lag("signature").over(w_ev), F.col("debut"))

livraisons_ev = livraisons_ev.withColumn(
    "duree_ev_min", (F.col("signature").cast("long") - depuis.cast("long")) / 60)

(livraisons_ev
    .filter(F.col("evenement").isNotNull())
    .groupBy("evenement")
    .agg(F.count("*").alias("nb"), F.round(F.avg("duree_ev_min"), 2).alias("duree_moy_min"))
    .show())

+---------+------+-------------+
|evenement|    nb|duree_moy_min|
+---------+------+-------------+
|    echec| 11220|         2.02|
|   livree|268682|         4.24|
+---------+------+-------------+



5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

In [21]:
par_livreur = (livraisons_ev
               .groupBy("livreur_id")
               .agg(F.count(F.when(F.col("evenement") == "livree", 1)).alias("nb_signees"),
                    F.count(F.when(F.col("evenement") == "echec", 1)).alias("nb_echecs"),
                    F.countDistinct("jour").alias("nb_jours"))
               .withColumn("signees_par_jour", F.round(F.col("nb_signees") / F.col("nb_jours"), 1)))

par_livreur.orderBy(F.desc("signees_par_jour")).show(10)

+----------+----------+---------+--------+----------------+
|livreur_id|nb_signees|nb_echecs|nb_jours|signees_par_jour|
+----------+----------+---------+--------+----------------+
|      L076|       468|       24|      15|            31.2|
|      L089|       804|       42|      26|            30.9|
|      L070|       543|       28|      18|            30.2|
|      L086|       713|       26|      24|            29.7|
|      L081|       615|       18|      21|            29.3|
|      L072|       490|       21|      17|            28.8|
|      L079|       459|       16|      16|            28.7|
|      L080|       457|       18|      16|            28.6|
|      L085|       715|       36|      25|            28.6|
|      L082|       451|       19|      16|            28.2|
+----------+----------+---------+--------+----------------+
only showing top 10 rows



## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

In [22]:
gps = gps.withColumn(
    "distance_km",
    F.coalesce(haversine(F.lag("lon").over(w_lj), F.lag("lat").over(w_lj), F.col("lon"), F.col("lat")),
               F.lit(0.0)))                     # 1er point de la journée : pas de point précédent -> 0
gps = cache_gps(gps)

gps.select("livreur_id", "timestamp", "dans_vehicule", "vitesse_kmh", "distance_km").show(5)

+----------+-------------------+-------------+-----------+--------------------+
|livreur_id|          timestamp|dans_vehicule|vitesse_kmh|         distance_km|
+----------+-------------------+-------------+-----------+--------------------+
|      L002|2025-05-09 07:41:00|        false|        0.0|                 0.0|
|      L002|2025-05-09 07:42:00|        false|        0.0|1.330398417274471E-4|
|      L002|2025-05-09 07:43:00|        false|        0.0|1.330398417274471E-4|
|      L002|2025-05-09 07:44:00|        false|        0.0|7.304305398644029E-5|
|      L002|2025-05-09 07:45:00|        false|        0.0|                 0.0|
+----------+-------------------+-------------+-----------+--------------------+
only showing top 5 rows



6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

In [23]:
km_vehicule = (gps.groupBy("vehicule_id")
               .agg(F.round(F.sum("distance_km"), 1).alias("km"))
               .withColumn("km_corrige", F.round(F.col("km") / 0.88, 1)))

km_vehicule.orderBy(F.desc("km")).show(10)
km_vehicule.agg(F.round(F.sum("km")).alias("km_flotte"),
                F.round(F.sum("km_corrige")).alias("km_flotte_corrige")).show()

+-----------+-------+----------+
|vehicule_id|     km|km_corrige|
+-----------+-------+----------+
|        V36|17442.2|   19820.7|
|        V35|17182.6|   19525.7|
|        V21|17031.3|   19353.8|
|        V20|16900.7|   19205.3|
|        V34|16864.6|   19164.3|
|        V19|16024.8|   18210.0|
|        V41|15516.1|   17631.9|
|        V40|15488.5|   17600.6|
|        V54|15379.7|   17476.9|
|        V53|15358.7|   17453.1|
+-----------+-------+----------+
only showing top 10 rows



+---------+-----------------+
|km_flotte|km_flotte_corrige|
+---------+-----------------+
| 521789.0|         592942.0|
+---------+-----------------+



6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

In [24]:
(gps.groupBy("livreur_id")
    .agg(F.round(F.sum("distance_km"), 1).alias("km"))
    .withColumn("km_corrige", F.round(F.col("km") / 0.88, 1))
    .orderBy(F.desc("km"))
    .show(10))

+----------+------+----------+
|livreur_id|    km|km_corrige|
+----------+------+----------+
|      L055|9734.6|   11062.0|
|      L056|9594.9|   10903.3|
|      L064|9511.1|   10808.1|
|      L057|9463.4|   10753.9|
|      L051|9446.3|   10734.4|
|      L050|9364.6|   10641.6|
|      L058|9318.7|   10589.4|
|      L047|9235.2|   10494.5|
|      L059|9225.8|   10483.9|
|      L060|9152.3|   10400.3|
+----------+------+----------+
only showing top 10 rows



6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

In [25]:
km_jour = (gps.groupBy("vehicule_id", "jour")
           .agg(F.sum("distance_km").alias("km"))
           .join(vehicule_entrepot, "vehicule_id"))

(km_jour.groupBy("entrepot")
    .agg(F.count("*").alias("nb_vehicule_jours"),
         F.round(F.avg("km"), 1).alias("km_moyen_par_jour"))
    .orderBy("entrepot")
    .show())

+--------+-----------------+-----------------+
|entrepot|nb_vehicule_jours|km_moyen_par_jour|
+--------+-----------------+-----------------+
|      E1|             3208|             45.8|
|      E2|             2967|             55.8|
|      E3|             4217|             49.6|
+--------+-----------------+-----------------+



6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

In [26]:
trajets = (gps.groupBy(*CLES)
           .agg(F.sum("distance_km").alias("km_jusqua_suivante"))
           .join(types.filter(F.col("type") == "livraison"), CLES))

trajets.agg(F.count("*").alias("nb_livraisons"),
            F.round(F.avg("km_jusqua_suivante"), 2).alias("moyenne_km"),
            F.round(F.median("km_jusqua_suivante"), 2).alias("mediane_km")).show()

+-------------+----------+----------+
|nb_livraisons|moyenne_km|mediane_km|
+-------------+----------+----------+
|       246258|      1.57|      0.84|
+-------------+----------+----------+



## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

In [27]:
ca_mois = (commandes
           .groupBy(F.date_format("horodatage", "yyyy-MM").alias("mois"))
           .agg(F.count("*").alias("nb_commandes"), F.round(F.sum("montant"), 2).alias("ca"))
           .orderBy("mois"))
ca_mois.show(12)

+-------+------------+----------+
|   mois|nb_commandes|        ca|
+-------+------------+----------+
|2025-01|       16186|3330814.25|
|2025-02|       15815|3251159.95|
|2025-03|       18006| 3775388.3|
|2025-04|       19008|3966928.45|
|2025-05|       20182| 4194991.2|
|2025-06|       21153|4410672.05|
|2025-07|       20803| 4226598.3|
|2025-08|       16490|3525075.45|
|2025-09|       26283| 5566650.7|
|2025-10|       25233| 5299971.4|
|2025-11|       31819| 6523520.5|
|2025-12|       40639| 8250067.4|
+-------+------------+----------+



7.2) Panier moyen : montant moyen d'une commande.

In [28]:
commandes.agg(F.round(F.avg("montant"), 2).alias("panier_moyen")).show()

+------------+
|panier_moyen|
+------------+
|      207.36|
+------------+



7.3) Panier médian.

In [29]:
commandes.agg(F.round(F.median("montant"), 2).alias("panier_median")).show()

+-------------+
|panier_median|
+-------------+
|        122.8|
+-------------+



7.4) Top 10 des produits par chiffre d'affaires.

In [30]:
ventes_produits = (lignes
                   .groupBy("produit_id", "nom", "categorie")
                   .agg(F.round(F.sum("montant"), 2).alias("ca"), F.sum("quantite").alias("quantite"))
                   .cache())
ventes_produits.orderBy(F.desc("ca")).show(10, truncate=False)

+----------+-----------------------+---------+---------+--------+
|produit_id|nom                    |categorie|ca       |quantite|
+----------+-----------------------+---------+---------+--------+
|P0096     |Café en grains 6x1 kg  |Boissons |2247675.0|32575   |
|P0049     |Lait demi-écrémé 6x1 L |Frais    |1621428.8|265808  |
|P0097     |Dosettes café x150     |Boissons |1384866.0|32973   |
|P0057     |Comté 12 mois 250 g    |Frais    |1300460.0|265400  |
|P0098     |Gobelets carton x1000  |Boissons |1141788.0|33582   |
|P0018     |Café en grains 1 kg    |Épicerie |1135793.4|88046   |
|P0083     |Eau gazeuse 6x1 L      |Boissons |1129588.2|289638  |
|P0150     |Cartouche d'encre noire|Maison   |1045053.0|41970   |
|P0065     |Pavés de saumon x2     |Frais    |1043281.9|132061  |
|P0068     |Oranges 2 kg           |Frais    |1038729.9|266341  |
+----------+-----------------------+---------+---------+--------+
only showing top 10 rows



7.5) Top 10 des produits par quantité vendue.

In [31]:
ventes_produits.orderBy(F.desc("quantite")).show(10, truncate=False)

+----------+---------------------+---------+---------+--------+
|produit_id|nom                  |categorie|ca       |quantite|
+----------+---------------------+---------+---------+--------+
|P0082     |Eau minérale 6x1,5 L |Boissons |958933.8 |290586  |
|P0084     |Eau de source 6x50 cl|Boissons |608567.4 |289794  |
|P0083     |Eau gazeuse 6x1 L    |Boissons |1129588.2|289638  |
|P0048     |Lait demi-écrémé 1 L |Frais    |283497.9 |269998  |
|P0070     |Carottes 1 kg        |Frais    |349476.4 |268828  |
|P0058     |Mozzarella 125 g     |Frais    |322360.8 |268634  |
|P0076     |Citrons x4           |Frais    |483445.8 |268581  |
|P0073     |Salade batavia       |Frais    |322122.0 |268435  |
|P0053     |Crème fraîche 20 cl  |Frais    |402519.0 |268346  |
|P0069     |Tomates grappe 1 kg  |Frais    |911427.8 |268067  |
+----------+---------------------+---------+---------+--------+
only showing top 10 rows



7.6) Top 10 des clients par dépense sur l'année.

In [32]:
(commandes.groupBy("client_id")
    .agg(F.round(F.sum("montant"), 2).alias("depense"), F.count("*").alias("nb_commandes"))
    .orderBy(F.desc("depense"))
    .show(10))

+---------+---------+------------+
|client_id|  depense|nb_commandes|
+---------+---------+------------+
|  CL00034| 436778.7|          20|
|  CL03089| 357505.4|          20|
|  CL02239| 244599.9|          64|
|  CL05203| 212285.5|          59|
|  CL04015| 204201.8|          60|
|  CL03099| 183802.7|          22|
|  CL07784|124598.05|          55|
|  CL03672|112956.85|          57|
|  CL05722|109450.85|          51|
|  CL02517|  92314.6|          54|
+---------+---------+------------+
only showing top 10 rows



7.7) Top 10 des clients par nombre de commandes.

In [33]:
(commandes.groupBy("client_id")
    .agg(F.count("*").alias("nb_commandes"), F.round(F.sum("montant"), 2).alias("depense"))
    .orderBy(F.desc("nb_commandes"), F.desc("depense"))
    .show(10))

+---------+------------+---------+
|client_id|nb_commandes|  depense|
+---------+------------+---------+
|  CL02239|          64| 244599.9|
|  CL03053|          61| 59428.75|
|  CL04015|          60| 204201.8|
|  CL05203|          59| 212285.5|
|  CL03672|          57|112956.85|
|  CL07784|          55|124598.05|
|  CL02517|          54|  92314.6|
|  CL06092|          53| 58813.55|
|  CL05722|          51|109450.85|
|  CL06965|          50|  7482.75|
+---------+------------+---------+
only showing top 10 rows



## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

In [34]:
date_cmd = (journal.filter(F.col("evenement") == "commande")
            .select("commande_id", F.to_date("horodatage").alias("date_commande")))
date_liv = (journal.filter(F.col("evenement") == "livree")
            .groupBy("commande_id").agg(F.min(F.to_date("horodatage")).alias("date_livraison")))

# left join : on garde aussi les commandes jamais livrées (date_livraison null), utile pour 8.2
delais = (date_cmd.join(date_liv, "commande_id", "left")
          .withColumn("delai_j", F.datediff("date_livraison", "date_commande"))
          .cache())

delais.filter(F.col("delai_j").isNotNull()).groupBy("delai_j").count().orderBy("delai_j").show()

+-------+------+
|delai_j| count|
+-------+------+
|      1|198280|
|      2| 53588|
|      3| 12435|
|      4|  3168|
|      5|   923|
|      6|   201|
|      7|    67|
|      8|     9|
|      9|     7|
|     10|     4|
+-------+------+



8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

In [35]:
(delais.groupBy(F.date_format("date_commande", "yyyy-MM").alias("mois"))
    .agg(F.count("*").alias("nb_commandes"),
         F.count("date_livraison").alias("nb_livrees"),
         F.count(F.when(F.col("delai_j") == 1, 1)).alias("nb_lendemain"))
    .withColumn("part_lendemain_pct", F.round(100 * F.col("nb_lendemain") / F.col("nb_commandes"), 1))
    .withColumn("part_parmi_livrees_pct", F.round(100 * F.col("nb_lendemain") / F.col("nb_livrees"), 1))
    .orderBy("mois")
    .show(12))

+-------+------------+----------+------------+------------------+----------------------+
|   mois|nb_commandes|nb_livrees|nb_lendemain|part_lendemain_pct|part_parmi_livrees_pct|
+-------+------------+----------+------------+------------------+----------------------+
|2025-01|       16186|     16119|       12870|              79.5|                  79.8|
|2025-02|       15815|     15749|       12573|              79.5|                  79.8|
|2025-03|       18006|     17936|       13759|              76.4|                  76.7|
|2025-04|       19008|     18925|       13863|              72.9|                  73.3|
|2025-05|       20182|     20080|       14093|              69.8|                  70.2|
|2025-06|       21153|     21049|       15913|              75.2|                  75.6|
|2025-07|       20803|     20714|       15877|              76.3|                  76.6|
|2025-08|       16490|     16434|       12408|              75.2|                  75.5|
|2025-09|       26283

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

In [36]:
suivi = (journal
         .groupBy("commande_id")
         .agg(F.max("client_id").alias("client_id"),
              F.count(F.when(F.col("evenement").isin("livree", "echec"), 1)).alias("nb_tentatives"),
              F.count(F.when(F.col("evenement") == "echec", 1)).alias("nb_echecs"),
              F.max((F.col("evenement") == "livree").cast("int")).alias("_livree"),
              F.max((F.col("evenement") == "retour").cast("int")).alias("_retour"))
         .withColumn("issue", F.when(F.col("_livree") == 1, "livree")
                               .when(F.col("_retour") == 1, "retour")
                               .otherwise("en cours"))
         .drop("_livree", "_retour")
         .cache())

suivi.groupBy("nb_tentatives").count().orderBy("nb_tentatives").show()
suivi.groupBy("issue").count().show()

+-------------+------+
|nb_tentatives| count|
+-------------+------+
|            0|  1663|
|            1|260399|
|            2|  9162|
|            3|   393|
+-------------+------+



+--------+------+
|   issue| count|
+--------+------+
|en cours|  1741|
|  livree|268682|
|  retour|  1194|
+--------+------+



8.4) Motifs d'échec : nombre d'échecs par motif.

In [37]:
(journal.filter(F.col("evenement") == "echec")
    .groupBy(F.col("detail").alias("motif"))
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

+----------------+-----+
|motif           |count|
+----------------+-----+
|absent          |8697 |
|acces_impossible|1372 |
|refus           |1151 |
+----------------+-----+



8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

In [38]:
taux_client = (suivi.groupBy("client_id")
               .agg(F.sum("nb_tentatives").alias("tentatives"), F.sum("nb_echecs").alias("echecs"))
               .withColumn("taux_echec", F.round(F.col("echecs") / F.col("tentatives"), 3))
               .cache())

(taux_client.filter(F.col("tentatives") >= 10)
    .orderBy(F.desc("taux_echec"), F.desc("tentatives"))
    .show(10))

+---------+----------+------+----------+
|client_id|tentatives|echecs|taux_echec|
+---------+----------+------+----------+
|  CL06349|        78|    39|       0.5|
|  CL05794|        73|    34|     0.466|
|  CL04500|        69|    32|     0.464|
|  CL01308|        74|    33|     0.446|
|  CL02052|        72|    32|     0.444|
|  CL04097|        67|    29|     0.433|
|  CL07039|        64|    25|     0.391|
|  CL00327|        67|    26|     0.388|
|  CL06732|        14|     4|     0.286|
|  CL07431|        16|     4|      0.25|
+---------+----------+------+----------+
only showing top 10 rows



## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*